In [ ]:
import sys
import json
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import FastEmbedEmbeddings

from src.config import (
    QDRANT_URL, 
    COLLECTION_NAME, 
    EMBEDDING_MODEL, 
    RETRIEVAL_TOP_K
)
from src.sidecar_manager import SidecarManager
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

# Inizializzazione Vector Store con FastEmbed nativo
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL)
vector_store = QdrantVectorStore(
    client=qdrant_client, 
    collection_name=COLLECTION_NAME, 
    embedding=embeddings
)


# Per il DataSet Pre-Taggato
sidecar_path = project_root / "data/processed/test/sidecar_04_05T.json"

# DataSet privo di Tag
#sidecar_path = project_root / "data/processed/test/sidecar_04_06T.json"

print("!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita")

In [5]:
# Caricamento della query di test da eval_queries.json
queries_path = project_root / "data/queries/ds1/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

matched = [q for q in benchmark_queries if q.get("id") == "Q_SINGLE_01"]
query_obj = matched[0]
query_text = query_obj["query"]

print(f"?> ID Query: {query_obj['id']}")
print(f"  >>> Testo Query: '{query_text}'\n")

# Vectorizzazione della query
query_vector = embeddings.embed_query(query_text)

# Retrieval vettoriale nativa con query_points
# with_vectors=True in modo da poterli passare al metodo
response = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=RETRIEVAL_TOP_K,
    with_vectors=True,
    with_payload=True
)
raw_records = response.points

print(f"!>> Recuperati {len(raw_records)} record con relativi vettori di embedding!")

?> ID Query: Q_SINGLE_01
  >>> Testo Query: 'Qual è la funzione dei reflection token nell'architettura Self-RAG?'

!>> Recuperati 7 record con relativi vettori di embedding!


In [6]:
builder = KnowledgeGraphBuilder()

### Popolamento dei nodi dai raw_records di Qdrant
for idx, rec in enumerate(raw_records):
    payload = rec.payload or {}
    
    # Costruzione dell'identificativo unico del chunk
    doc_id = payload.get("doc_id", "doc")
    chunk_idx = payload.get("chunk_index", idx)
    chunk_id = payload.get("chunk_id", f"{doc_id}_chunk_{chunk_idx}")
    
    # Estrazione testo e tag
    text = payload.get("text", payload.get("page_content", ""))
    tags = payload.get("user_tags", payload.get("tags", []))
    
    # Aggiunta del nodo con salvataggio automatico del vettore
    builder.add_chunk_node(
        chunk_id=chunk_id,
        text=text,
        vector=rec.vector,
        tags=tags
    )

### Calcolo automatico degli archi tramite Cosine Similarity (soglia da config)
builder.auto_connect_nodes()

### Generazione del JSON per la visualizzazione
graph_data = builder.to_json_data()

print(f"!>> Grafo generato con successo:")
print(f"   >>> Nodi inseriti: {len(graph_data['nodes'])}")
print(f"   >>> Archi collegati: {len(graph_data['links'])}")

### Inizializzazione e rendering del Widget
widget = ChunkGraphWidget(sidecar_path=str(sidecar_path))
widget.load_graph(graph_data)

widget

!>> Grafo generato con successo:
   >>> Nodi inseriti: 7
   >>> Archi collegati: 16


In [ ]:
from src.tag_assigner import TagAssigner
from src.tag_reranker import TagReranker

tag_assigner = TagAssigner()
tag_reranker = TagReranker(sidecar_manager=widget.sidecar, tag_assigner=tag_assigner)

# Reranking
reranked_results = tag_reranker.rerank(
    query_text=query_text,
    retrieved_points=raw_records
) 

print(f"!>> Reranking completato. Elaborati {len(reranked_results)} chunk candidati.\n")

if reranked_results:
    print(f"  >>> Tag assegnati alla Query: {reranked_results[0].get('query_tags', [])}")
    print("\n--- Dettaglio Punteggi Post-Reranking ---")
    for item in reranked_results:
        print(f"  > ID: {item['chunk_id']}")
        print(f"    Score Qdrant: {item['initial_score']} | Score Finale: {item['final_score']} (Moltiplicatore W_tag: {item['weight_factor']})")
        print(f"    Tag Chunk: {item['chunk_tags']} | Tag Coincidenti: {item['matched_tags']}\n")

# Ricostruzione del grafo con i metadati di reranking
builder_reranked = KnowledgeGraphBuilder()

for item in reranked_results:
    payload = item.get("payload", {})
    text = payload.get("text", payload.get("page_content", ""))
    
    builder_reranked.add_chunk_node(
        chunk_id=item["chunk_id"],
        text=text,
        vector=item["vector"],
        tags=item["chunk_tags"],
        metadata={
            "initial_score": item.get("initial_score"),
            "final_score": item.get("final_score"),
            "weight_factor": item.get("weight_factor"),
            "query_tags": item.get("query_tags"),
            "matched_tags": item.get("matched_tags")
        }
    ) 

builder_reranked.auto_connect_nodes()
graph_data_reranked = builder_reranked.to_json_data()

# DEBUG print
print(f"!>> Grafo aggiornato post-reranking:")
print(f"   >>> Nodi: {len(graph_data_reranked['nodes'])}")
print(f"   >>> Archi: {len(graph_data_reranked['links'])}")

print("\n--- Analisi Impatto Reranking (Score & Moltiplicatori) ---")
for item in reranked_results:
    init_s = item["initial_score"]
    final_s = item["final_score"]
    delta_s = final_s - init_s
    w_factor = item["weight_factor"]
    matched = item["matched_tags"]
    
    # Formattazione per allineare le colonne nei log
    print(
        f"Chunk: {item['chunk_id']:<22} | "
        f"Score: {init_s:.4f} -> {final_s:.4f} (Δ: {delta_s:+.4f}) | "
        f"W_tag: {w_factor:.2f}x | Match: {matched}"
    )
    
# Aggiornamento reattivo istanza widget preesistente
#! NB: è fondamentale usare il widget preesistente e non reistanziarlo
#|     per evitare di avere 2 copie di SidecarManager che puntano allo stesso .json
#!     incombendo così nella problematica delle 2 cache 
widget.load_graph(graph_data_reranked)
widget